In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Define symbolic variables

n = sp.symbols('n', integer=True)
A_1, A_2 = sp.symbols('A_1 A_2')

display(Markdown("### Analytical Symbolic Output"))

# Coefficients of the difference equation:
# alpha_0 = 1, alpha_1 = sqrt(3), alpha_2 = 1

alpha = [1.0, np.sqrt(3), 1.0]

# Automatically compute the roots of the characteristic polynomial

roots = np.roots([alpha[0], alpha[1], alpha[2]])

display(Markdown( "**Computed Characteristic Roots ($\\lambda$):** " f"$\\lambda_1 = {roots[0].real:.3f}" f"{roots[0].imag:+.3f}j$, "
                   f"$\\lambda_2 = {roots[1].real:.3f}" f"{roots[1].imag:+.3f}j$" ))

# Setup initial conditions / Vandermonde system for A1, A2
# using the computed roots

eq1 = sp.Eq(A_1 + A_2, 1.0)
eq2 = sp.Eq(A_1 * (roots[0]**(-1)) + A_2 * (roots[1]**(-1)), 0.0 )
sol = sp.solve((eq1, eq2), (A_1, A_2))

# Round coefficients to 3 decimal places

a1_val = complex(round(float(sp.re(sol[A_1])), 3), round(float(sp.im(sol[A_1])), 3))
a2_val = complex(round(float(sp.re(sol[A_2])), 3), round(float(sp.im(sol[A_2])), 3))

r1_val = complex(round(float(roots[0].real), 3), round(float(roots[0].imag), 3))
r2_val = complex(round(float(roots[1].real), 3), round(float(roots[1].imag), 3))

# Display constants A1 and A2 rounded to 3 decimal places

display(Markdown(f"**Calculated Constants:** " f"$A_1 = {a1_val.real:.3f}" f"{a1_val.imag:+.3f}j$, "
    f"$A_2 = {a2_val.real:.3f}" f"{a2_val.imag:+.3f}j$" ))

# Intermediate response h_beta[n]

r1_sym = sp.N (sp.Float(r1_val.real) + sp.I * sp.Float(r1_val.imag), 4)
r2_sym = sp.N (sp.Float(r2_val.real) + sp.I * sp.Float(r2_val.imag), 4)
a1_sym = sp.N (sp.Float(a1_val.real) + sp.I * sp.Float(a1_val.imag), 4)
a2_sym = sp.N (sp.Float(a2_val.real) + sp.I * sp.Float(a2_val.imag), 4)
h_beta = (a1_sym * r1_sym**n + a2_sym * r2_sym**n ) * sp.Heaviside(n)

# Final impulse response
# h[n] = h_beta[n-1] + h_beta[n-2]

h_final = (a1_sym * r1_sym**(n - 1) + a2_sym * r2_sym**(n - 1)) * sp.Heaviside(n - 1) + (a1_sym * r1_sym**(n - 2) + a2_sym * r2_sym**(n - 2)) * sp.Heaviside(n - 2)
display(Markdown("**Intermediate Response ($h_{\\beta}[n]$):**"))
display(sp.Eq(sp.Symbol('h_beta[n]'), h_beta ))
display(Markdown("**Final Impulse Response $h[n]$: **"))
display(sp.Eq(sp.Symbol('h[n]'), h_final))

# Numerical evaluation for plotting
n_vals = np.arange(0, 16)
h_beta_vals = (a1_val * (roots[0]**n_vals) + a2_val * (roots[1]**n_vals)) * (n_vals >= 0)

# Remove negligible numerical imaginary parts
h_beta_vals = np.real(h_beta_vals)
h_final_vals = np.zeros_like(n_vals, dtype=float)
for idx, val in enumerate(n_vals):
    if val >= 1:
        n_sub = val - 1
        h_final_vals[idx] += np.real(a1_val * (roots[0]**n_sub) + a2_val * (roots[1]**n_sub))
    if val >= 2:
        n_sub = val - 2
        h_final_vals[idx] += np.real(a1_val * (roots[0]**n_sub) + a2_val * (roots[1]**n_sub))

# Plotting the responses
fig, axes = plt.subplots( 2, 1, figsize=(10, 6), sharex=True )
axes[0].stem( n_vals, h_beta_vals, linefmt='b-', markerfmt='bo', basefmt='k-' )
axes[0].set_title(r'Intermediate Response $h_{\beta}[n]$', fontsize=9.5, fontweight='bold', color='darkblue' )
axes[0].set_ylabel ('Amplitude', fontsize=8.5)
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[1].stem(n_vals, h_final_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
axes[1].set_title( r'Final System Impulse Response $h[n]$', fontsize=9.5, fontweight='bold', color='darkred' )
axes[1].set_xlabel('Index $n$', fontsize=8.5 )
axes[1].set_ylabel( 'Amplitude', fontsize=8.5 )
axes[1].grid(True, linestyle='--', alpha=0.6 )
plt.tight_layout()
plt.show()
